In [42]:
import pandas as pd

# 1. Memuat dataset
df = pd.read_csv('Amazon Sale Report.csv', low_memory=False)


In [43]:
# 2. Membersihkan dan konversi tanggal dengan format 'mixed'
df['Date'] = pd.to_datetime(df['Date'], format='mixed', errors='coerce')
df['Amount'] = df['Amount'].fillna(0)
df['Wilayah'] = df['ship-state'].fillna('Lainnya').str.title()

In [44]:
# 3. Menghapus baris tanpa tanggal dan mengurutkannya
df = df.dropna(subset=['Date']).sort_values('Date')

# Cek hasil pembersihan data
print("Data berhasil dimuat dan dibersihkan tanpa warning!")
df[['Date', 'Wilayah', 'Amount', 'Qty']].head()

Data berhasil dimuat dan dibersihkan tanpa warning!


,Date,Wilayah,Amount,Qty
48970,2022-03-31,West Bengal,495.00,1
49001,2022-03-31,Karnataka,599.00,1
49002,2022-03-31,Haryana,416.19,0
49003,2022-03-31,Maharashtra,449.00,1
49004,2022-03-31,Telangana,542.00,1


In [45]:
import plotly.express as px

# 1. Agregasi data penjualan harian untuk seluruh wilayah (Overview First)
trend_df = df.groupby('Date')['Amount'].sum().reset_index()

# Hitung persentase perubahan dari hari ke hari
trend_df['Pct_Change'] = trend_df['Amount'].pct_change() * 100
trend_df['Pct_Change'] = trend_df['Pct_Change'].fillna(0)

In [46]:
# 2. Membuat Grafik Garis Interaktif
fig = px.line(
    trend_df,
    x='Date',
    y='Amount',
    title="<b>Overview First: Tren Total Penjualan Seluruh Wilayah</b>",
    labels={'Amount': 'Total Penjualan (INR)', 'Date': 'Tanggal Transaksi'}
)

In [47]:
# 3. Details-on-Demand (Hover Tooltip Kustom)
fig.update_traces(
    mode='lines+markers',
    line=dict(color='#1f77b4', width=2),
    marker=dict(size=4),
    hovertemplate=(
        "<b>Tanggal:</b> %{x|%d %B %Y}<br>" +
        "<b>Total Penjualan:</b> INR %{y:,.2f}<br>" +
        "<b>Perubahan:</b> %{customdata:.2f}%<br>" +
        "<extra></extra>"
    ),
    customdata=trend_df['Pct_Change']
)

In [48]:
# 4. Penerapan Data-Ink Ratio (Tampilan Bersih & Minimalis)
fig.update_layout(
    template='plotly_white',
    xaxis_showgrid=False,
    yaxis_showgrid=True,
    yaxis_gridcolor='#EAEAEA',
    title_font=dict(size=16)
)

# Tampilkan Grafik
fig.show()

In [49]:
import ipywidgets as widgets
from IPython.display import display, HTML
import plotly.express as px
from google.colab import output

# Mengaktifkan manager widget bawaan Colab agar Plotly langsung dirender
output.enable_custom_widget_manager()

# 1. Menyiapkan pilihan wilayah
daftar_wilayah = ['Semua Wilayah (Overview)'] + sorted(df['Wilayah'].unique().tolist())

# 2. Fungsi pembuat grafik
def buat_grafik_penjualan(Wilayah):
    if Wilayah == 'Semua Wilayah (Overview)':
        filtered_df = df.copy()
        judul = "Overview First: Tren Total Penjualan Seluruh Wilayah"
    else:
        filtered_df = df[df['Wilayah'] == Wilayah]
        judul = f"Zoom & Filter: Tren Penjualan Wilayah {Wilayah}"

    # Agregasi data harian
    trend_df = filtered_df.groupby('Date')['Amount'].sum().reset_index()
    trend_df['Pct_Change'] = trend_df['Amount'].pct_change().fillna(0) * 100

    # Membuat Grafik Plotly
    fig = px.line(
        trend_df,
        x='Date',
        y='Amount',
        title=f"<b>{judul}</b>",
        labels={'Amount': 'Total Penjualan (INR)', 'Date': 'Tanggal Transaksi'}
    )

    # Details-on-Demand (Hover Tooltip Kustom)
    fig.update_traces(
        mode='lines+markers',
        line=dict(color='#1f77b4', width=2),
        marker=dict(size=4),
        hovertemplate=(
            "<b>Tanggal:</b> %{x|%d %B %Y}<br>" +
            "<b>Total Penjualan:</b> INR %{y:,.2f}<br>" +
            "<b>Perubahan:</b> %{customdata:.2f}%<br>" +
            "<extra></extra>"
        ),
        customdata=trend_df['Pct_Change']
    )

  # Data-Ink Ratio
    fig.update_layout(
        template='plotly_white',
        xaxis_showgrid=False,
        yaxis_showgrid=True,
        yaxis_gridcolor='#EAEAEA',
        height=450,
        margin=dict(l=40, r=40, t=50, b=40)
    )

    return fig

# 3. Tampilan Judul dan Widget Interaktif
display(HTML("<h3>Dasbor Eksekutif Penjualan Interaktif - PT Maju Sejahtera</h3>"))
widgets.interact(buat_grafik_penjualan, Wilayah=daftar_wilayah);

interactive(children=(Dropdown(description='Wilayah', options=('Semua Wilayah (Overview)', 'Andaman & Nicobar …